In [1]:
## reload submodule
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

import qfin

In [ ]:
data = pd.read_csv("./data/_^spx.csv", index_col=0, parse_dates=[0], sep=",")
data

In [4]:
backtest_params = {
    "initial_balance": 10000,
    "default_entry_value": 1,
    "default_entry_value_max": 1000000,
    "commission": 0.001,
}

bt = qfin.Backtester(dataset=data, **backtest_params)

for broker in bt.run():
    current_bar = broker.state.data.iloc[-1]
    previous_bar = broker.state.data.iloc[-2]

    current_signal = current_bar["signal"]
    previous_signal = previous_bar["signal"]
    changed = current_signal != previous_signal

    if changed:
        if current_signal == 1:
            broker.buy()
        elif current_signal == -1:
            broker.sell()
        else:
            broker.close()

In [5]:
trades = bt.trades()
trades.columns

Index(['is_long', 'leverage', 'entry_value', 'entry_price', 'entry_bar',
       'entry_commission', 'entry_time', 'exit_value', 'exit_price',
       'exit_commission', 'exit_bar', 'exit_time', 'pnl', 'return_pct'],
      dtype='object')

In [6]:
trades[["is_long", "entry_time", "entry_price", "exit_price", "pnl", "return_pct"]]

,is_long,entry_time,entry_price,exit_price,pnl,return_pct
0,False,2023-02-02,4179.759766,3861.590088,823.110431,0.082393
1,True,2023-03-10,3861.590088,4588.959961,2034.570360,0.188360
2,False,2023-07-31,4588.959961,4117.370117,1468.497034,0.114537
3,True,2023-10-27,4117.370117,5254.350098,3941.641614,0.276142
4,False,2024-03-28,5254.350098,4967.229980,1051.633991,0.057803
5,True,2024-04-19,4967.229980,5667.200195,2709.115261,0.140918
6,False,2024-07-16,5667.200195,5186.330078,2031.404040,0.092719
7,True,2024-08-05,5186.330078,5648.399902,2130.656820,0.089094
8,False,2024-08-30,5648.399902,5408.419922,1154.426476,0.044372
9,True,2024-09-06,5408.419922,6090.270020,3422.020296,0.126072


In [7]:
history = bt.history()
history.columns

Index(['close', 'balance', 'equity', 'commission', 'long', 'short', 'signal',
       'buy_hold'],
      dtype='object')

In [8]:
history.tail(20)

,close,balance,equity,commission,long,short,signal,buy_hold
date,,,,,,,,
2025-03-11,5572.069824,32995,36346,806,False,True,-1,14570.779262
2025-03-12,5599.299805,32995,36169,806,False,True,-1,14641.984765
2025-03-13,5521.520020,32995,36679,806,True,False,1,14438.593186
2025-03-14,5638.939941,36675,37418,879,True,False,1,14745.642418
2025-03-17,5675.120117,36675,37658,879,True,False,1,14840.252388
2025-03-18,5614.660156,36675,37257,879,True,False,1,14682.151579
2025-03-19,5675.290039,36675,37659,879,True,False,1,14840.696728
2025-03-20,5662.890137,36675,37577,879,True,False,1,14808.271391
2025-03-21,5667.560059,36675,37608,879,True,False,1,14820.483083


In [9]:
stats = bt.stats()
stats

Start                     2023-01-03 00:00:00
End                       2025-04-07 00:00:00
Duration                    825 days 00:00:00
Exposure Time [%]                   96.296296
Equity Start                            10000
Equity Peak                             43690
Equity Final                            43690
Equity Return [%]                       336.9
Balance Start                           10000
Balance Peak                            43690
Balance Final                           43690
Balance Return [%]                      336.9
Gross Return [%]                       159.71
Total Commissions                        1000
Return (Ann.) [%]                   92.804468
Volatility (Ann.) [%]               25.731644
CAGR [%]                            92.094352
Sharpe Ratio                         3.412315
Sortino Ratio                       14.088748
Calmar Ratio                        27.526715
Max. Drawdown [%]                   -3.371433
Avg. Drawdown [%]                 

In [ ]:
bt.plot(w=700, h=500, show_signals=False)